# LAFigureSpecs Binder demo

`LAFigureSpecs` computes linear algebra figure specifications and delegates layout/rendering to `matrixlayout`. This notebook exercises the main interactive paths.

In [ ]:
import sympy as sym
from IPython.display import SVG, display

import LAFigureSpecs as lf

RENDER_OPTS = dict(
    toolchain_name="pdftex_dvisvgm",
    crop="tight",
    padding=(2, 2, 2, 2),
    output_dir="/tmp/la/run",
)

def show(svg):
    display(SVG(svg))

## Top-level use functions

| Area | Function | Output |
|---|---|---|
| GE | `ge(...)` | Rendered SVG for a row-reduction layout. |
| GE | `show_ge(...)` / `ShowGE` | Notebook-friendly row-reduction workflow/display helpers. |
| GE | `lhs_matrix(show)` / `rhs_matrix(show)` / `rhs_column(show, ...)` | Direct accessors for the current GE stack state. |
| GE | `ge_spec(...)` | Reusable spec dictionary for a GE layout. |
| GE | `ge_tex(...)` | TeX source for a GE layout. |
| GE | `ge_table_svg(...)` | Rendered SVG for a GE layout. |
| GE | `ge_bundle(...)` | `spec`, `tex`, optional `svg`, and render status in one object. |
| QR | `qr_figure(...)` | Computes and renders a Gram-Schmidt/QR layout as SVG. |
| QR | `qr(...)` | Renders a precomputed QR matrix stack as SVG. |
| QR | `qr_spec(...)` | Reusable spec dictionary for a QR layout. |
| QR | `qr_tex(...)` | TeX source for a QR layout. |
| QR | `qr_table_svg(...)` | Rendered SVG for a QR layout. |
| QR | `qr_bundle(...)` | `spec`, `tex`, optional `svg`, and render status in one object. |
| Eigen | `eigendecomposition(...)` | Structured eigenvalue/eigenvector data. |
| Eigen | `eig_spec(...)` | Reusable spec dictionary for an eigen layout. |
| Eigen | `eig_tex(...)` | TeX source for an eigen layout. |
| Eigen | `eig_svg(...)` | Rendered SVG for an eigen layout. |
| Eigen | `eig_bundle(...)` | `spec`, `tex`, optional `svg`, and render status in one object. |

## Row reduction

In [ ]:
A = sym.Matrix([[1, 2], [1, 2], [3, 4]])
spec = lf.ge_spec(A, gj=False, array_names=["E", "A"])
spec["outer_hspace_mm"] = 8
spec["cell_align"] = "r"

svg = lf.render_ge_svg(**spec, **RENDER_OPTS)
show(svg)

## Gaussian elimination with a right-hand side

In [ ]:
A = sym.Matrix([[-2, 0, -2, 2, 0], [2, 0, 3, -3, 0], [0, 0, 0, 0, 1]])
b = sym.Matrix([4, -5, -1])

svg = lf.ge_table_svg(A, rhs=b, gj=False, array_names=["E", ["A", "b"]], fig_scale=1.1, **RENDER_OPTS)
show(svg)

## Display system of equations and solution

In [ ]:
U = sym.Matrix([[-2, 0, -2, 2, 0], [0, 0, 1, -1, 0], [0, 0, 0, 0, 1]])
rhs = sym.Matrix([4, -1, -1])

cascade_tex = "\n".join(lf.backsubstitution_tex(U, rhs))

show(lf.latex_svg("$" + r"(\xi) \Leftrightarrow" + lf.linear_system_tex(U, rhs) + "$", scale=1.1, **RENDER_OPTS))
show(lf.latex_svg(cascade_tex, scale=1.1, **RENDER_OPTS))
show(lf.latex_svg(lf.standard_solution_tex(U, rhs), scale=1.1, **RENDER_OPTS))

## Gram-Schmidt / QR layout

In [ ]:
A = sym.Matrix([[-1, -4, -3], [1, 0, -3], [1, 4, 1], [-1, 0, 1]])

svg = lf.qr_figure(A, fig_scale=1.1, **RENDER_OPTS)
show(svg)

## Eigenproblem table

In [ ]:
A = sym.Matrix([[2, 1], [0, 3]])

svg = lf.eig_svg(A, fig_scale=1.1, **RENDER_OPTS)
show(svg)

## SVD table

In [ ]:
A = sym.Matrix([
    [sym.Rational(6, 35), 0, sym.Rational(-8, 35), sym.Rational(3, 7)],
    [sym.Rational(18, 35), 0, sym.Rational(-24, 35), sym.Rational(2, 7)],
    [sym.Rational(9, 35), 0, sym.Rational(-12, 35), sym.Rational(-6, 7)],
])

svg = lf.svd_svg(A, fig_scale=1.1, **RENDER_OPTS, factor_out={"u": True, "v": True})
show(svg)

## Low-level LaTeX rendering with another package

`LAFigureSpecs` uses packages such as TikZ, `systeme`, `cascade`, and `nicematrix` for its own figures. For `tikz-cd`, the most robust path is to render a full LaTeX document with `lf.latex_document_svg(...)`.

In [ ]:
tikzcd_tex = r"""
\begin{tikzcd}[column sep=large, row sep=large, ampersand replacement=\&]
\mathbb{R}^n \arrow[r, "T"] \arrow[d, "P"'] \&
\mathbb{R}^m \arrow[d, "Q"] \\
\mathbb{R}^n \arrow[r, "A_T"'] \&
\mathbb{R}^m
\end{tikzcd}
"""

tikzcd_doc = rf"""\documentclass{{standalone}}
\usepackage{{amsmath,amssymb,tikz-cd}}
\begin{{document}}
\scalebox{{1.1}}{{%
{tikzcd_tex}
}}
\end{{document}}
"""

show(lf.latex_document_svg(tikzcd_doc, toolchain_name="pdftex_pdftocairo", crop=RENDER_OPTS["crop"], padding=RENDER_OPTS["padding"], output_dir=RENDER_OPTS["output_dir"], output_stem="tikzcd"))